In [0]:
%sql
use catalog deltalake_catalog

In [0]:
%sql
create volume if not exists deltalake_catalog.default.delta_volume1

In [0]:
dbutils.fs.mkdirs("/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1")

True

In [0]:
%fs mkdirs /Volumes/deltalake_catalog/default/delta_volume1/ordersdata2

True

In [0]:
%fs ls /Volumes/deltalake_catalog/default/delta_volume1

path,name,size,modificationTime
dbfs:/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1/,ordersdata1/,0,1756407762000
dbfs:/Volumes/deltalake_catalog/default/delta_volume1/ordersdata2/,ordersdata2/,0,1756407859000


In [0]:
# Orders sample data
data = [
    (1, "SKU-1001", "Wireless Mouse", "Electronics", 2, 799.00),
    (2, "SKU-2001", "Yoga Mat", "Fitness", 1, 1199.00),
    (3, "SKU-3001", "Notebook A5", "Stationery", 5, 49.50),
    (4, "SKU-4001", "Coffee Mug", "Kitchen", 3, 299.00),
    (5, "SKU-5001", "LED Bulb", "Electronics", 4, 149.99)
]

# Define schema (column names)
columns = ["order_id", "sku", "product_name", "product_category", "qty", "unit_price"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Display DataFrame
display(df)

order_id,sku,product_name,product_category,qty,unit_price
1,SKU-1001,Wireless Mouse,Electronics,2,799.0
2,SKU-2001,Yoga Mat,Fitness,1,1199.0
3,SKU-3001,Notebook A5,Stationery,5,49.5
4,SKU-4001,Coffee Mug,Kitchen,3,299.0
5,SKU-5001,LED Bulb,Electronics,4,149.99


In [0]:
volume_path = "/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1"
df.write.format("delta").mode("overwrite").save(volume_path)

In [0]:
%sql
SELECT * FROM DELTA.`/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1`


order_id,sku,product_name,product_category,qty,unit_price
1,SKU-1001,Wireless Mouse,Electronics,2,799.0
2,SKU-2001,Yoga Mat,Fitness,1,1199.0
3,SKU-3001,Notebook A5,Stationery,5,49.5
4,SKU-4001,Coffee Mug,Kitchen,3,299.0
5,SKU-5001,LED Bulb,Electronics,4,149.99


In [0]:
%sql
DESCRIBE DETAIL DELTA.`/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1`

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,af5776de-5954-4c15-9b77-7aa41fe42c05,null,null,dbfs:/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1,2025-08-28T19:08:15.078Z,2025-08-28T19:08:18.000Z,List(),List(),1,1973,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE FORMATTED DELTA.`/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1`

col_name,data_type,comment
order_id,bigint,null
sku,string,null
product_name,string,null
product_category,string,null
qty,bigint,null
unit_price,double,null
,,
# Delta Statistics Columns,,
Column Names,"unit_price, product_name, qty, product_category, order_id, sku",
Column Selection Method,first-32,


In [0]:
%fs ls /Volumes/deltalake_catalog/default/delta_volume1/ordersdata1/

path,name,size,modificationTime
dbfs:/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1/_delta_log/,_delta_log/,0,1756408095000
dbfs:/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1/part-00000-e6c888f9-f3ad-41f1-919c-70fe185eda24.c000.snappy.parquet,part-00000-e6c888f9-f3ad-41f1-919c-70fe185eda24.c000.snappy.parquet,1973,1756408098000


In [0]:
log_file_path = "dbfs:/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1/_delta_log/00000000000000000000.json"
df_log = spark.read.json(log_file_path)
display(df_log)

add,commitInfo,metaData,protocol
null,"List(0828-185806-g98h13g2-v2n, Databricks-Runtime/17.1.x-photon-scala2.13, true, WriteSerializable, List(399050216140134), WRITE, List(1, 1973, 5), List(ErrorIfExists, [], false), List(true, false), 1756408098601, 9eb15dc5-72cd-46ab-9737-8bbc0af833cf, 147711536460494, sumit.trendytech03@outlook.com)",null,null
null,null,"List(List(true), 1756408095078, List(parquet), af5776de-5954-4c15-9b77-7aa41fe42c05, List(), {""type"":""struct"",""fields"":[{""name"":""order_id"",""type"":""long"",""nullable"":true,""metadata"":{}},{""name"":""sku"",""type"":""string"",""nullable"":true,""metadata"":{}},{""name"":""product_name"",""type"":""string"",""nullable"":true,""metadata"":{}},{""name"":""product_category"",""type"":""string"",""nullable"":true,""metadata"":{}},{""name"":""qty"",""type"":""long"",""nullable"":true,""metadata"":{}},{""name"":""unit_price"",""type"":""double"",""nullable"":true,""metadata"":{}}]})",null
null,null,null,"List(3, 7, List(deletionVectors), List(deletionVectors, appendOnly, invariants))"
"List(true, 1756408098000, part-00000-e6c888f9-f3ad-41f1-919c-70fe185eda24.c000.snappy.parquet, 1973, {""numRecords"":5,""minValues"":{""order_id"":1,""sku"":""SKU-1001"",""product_name"":""Coffee Mug"",""product_category"":""Electronics"",""qty"":1,""unit_price"":49.5},""maxValues"":{""order_id"":5,""sku"":""SKU-5001"",""product_name"":""Yoga Mat"",""product_category"":""Stationery"",""qty"":5,""unit_price"":1199.0},""nullCount"":{""order_id"":0,""sku"":0,""product_name"":0,""product_category"":0,""qty"":0,""unit_price"":0},""tightBounds"":true}, List(1756408098000000, 1756408098000000, 1756408098000000, 268435456))",null,null,null


In [0]:
%sql
SELECT * FROM JSON.`dbfs:/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1/_delta_log/00000000000000000000.json`

add,commitInfo,metaData,protocol
null,"List(0828-185806-g98h13g2-v2n, Databricks-Runtime/17.1.x-photon-scala2.13, true, WriteSerializable, List(399050216140134), WRITE, List(1, 1973, 5), List(ErrorIfExists, [], false), List(true, false), 1756408098601, 9eb15dc5-72cd-46ab-9737-8bbc0af833cf, 147711536460494, sumit.trendytech03@outlook.com)",null,null
null,null,"List(List(true), 1756408095078, List(parquet), af5776de-5954-4c15-9b77-7aa41fe42c05, List(), {""type"":""struct"",""fields"":[{""name"":""order_id"",""type"":""long"",""nullable"":true,""metadata"":{}},{""name"":""sku"",""type"":""string"",""nullable"":true,""metadata"":{}},{""name"":""product_name"",""type"":""string"",""nullable"":true,""metadata"":{}},{""name"":""product_category"",""type"":""string"",""nullable"":true,""metadata"":{}},{""name"":""qty"",""type"":""long"",""nullable"":true,""metadata"":{}},{""name"":""unit_price"",""type"":""double"",""nullable"":true,""metadata"":{}}]})",null
null,null,null,"List(3, 7, List(deletionVectors), List(deletionVectors, appendOnly, invariants))"
"List(true, 1756408098000, part-00000-e6c888f9-f3ad-41f1-919c-70fe185eda24.c000.snappy.parquet, 1973, {""numRecords"":5,""minValues"":{""order_id"":1,""sku"":""SKU-1001"",""product_name"":""Coffee Mug"",""product_category"":""Electronics"",""qty"":1,""unit_price"":49.5},""maxValues"":{""order_id"":5,""sku"":""SKU-5001"",""product_name"":""Yoga Mat"",""product_category"":""Stationery"",""qty"":5,""unit_price"":1199.0},""nullCount"":{""order_id"":0,""sku"":0,""product_name"":0,""product_category"":0,""qty"":0,""unit_price"":0},""tightBounds"":true}, List(1756408098000000, 1756408098000000, 1756408098000000, 268435456))",null,null,null


In [0]:
%sql
SELECT * FROM PARQUET.`dbfs:/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1/part-00000-e6c888f9-f3ad-41f1-919c-70fe185eda24.c000.snappy.parquet`

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5464448193142474>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SELECT * FROM PARQUET.`dbfs:/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1/part-00000-e6c888f9-f3ad-41f1-919c-70fe185eda24.c000.snappy.parquet`\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2543, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2541 with self.builtin_trap:
   2542     args = (magic_arg_s, cell)
-> 2543     result = fn(*args, **kwargs)
   2545 # The code below prevents the output from being displayed
   2546 # when using magics with decorator @output_can_be_silenced
   2547 # when the last Python token in the expression is a ';'.
   2548 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_mag

In [0]:

# Orders sample data
data = [
    (6, "SKU-6001", "Running Shoes", "Fitness", 1, 2599.00),
    (7, "SKU-7001", "Desk Chair", "Furniture", 1, 5499.00)
]

# Define schema (column names)
columns = ["order_id", "sku", "product_name", "product_category", "qty", "unit_price"]

# Create DataFrame
new_df = spark.createDataFrame(data, columns)

# Display DataFrame
display(new_df)

order_id,sku,product_name,product_category,qty,unit_price
6,SKU-6001,Running Shoes,Fitness,1,2599.0
7,SKU-7001,Desk Chair,Furniture,1,5499.0


In [0]:
volume_path = "/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1"
new_df.write.format("delta").mode("append").save(volume_path)

In [0]:
%sql
SELECT * FROM JSON.`dbfs:/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1/_delta_log/00000000000000000001.json`

add,commitInfo
null,"List(0828-185806-g98h13g2-v2n, Databricks-Runtime/17.1.x-photon-scala2.13, true, WriteSerializable, List(399050216140134), WRITE, List(1, 1829, 2), List(Append, [], false), 0, List(true, false), 1756409114913, 55e2bc68-8d68-4e8d-b2ca-e59204ac8bc5, 147711536460494, sumit.trendytech03@outlook.com)"
"List(true, 1756409114000, part-00000-33be1548-939f-4562-8ce5-92b3b0e5b8f4.c000.snappy.parquet, 1829, {""numRecords"":2,""minValues"":{""order_id"":6,""sku"":""SKU-6001"",""product_name"":""Desk Chair"",""product_category"":""Fitness"",""qty"":1,""unit_price"":2599.0},""maxValues"":{""order_id"":7,""sku"":""SKU-7001"",""product_name"":""Running Shoes"",""product_category"":""Furniture"",""qty"":1,""unit_price"":5499.0},""nullCount"":{""order_id"":0,""sku"":0,""product_name"":0,""product_category"":0,""qty"":0,""unit_price"":0},""tightBounds"":true}, List(1756409114000000, 1756409114000000, 1756409114000000, 268435456))",null


In [0]:
%sql
describe history DELTA.`/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2025-08-28T19:25:14.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(399050216140134),0828-185806-g98h13g2-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1829)",null,Databricks-Runtime/17.1.x-photon-scala2.13
0,2025-08-28T19:08:18.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> ErrorIfExists, statsOnLoad -> false, partitionBy -> [])",null,List(399050216140134),0828-185806-g98h13g2-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 1973)",null,Databricks-Runtime/17.1.x-photon-scala2.13
